# Data Cleaning — Sri Lanka Property Price Predictor

This notebook handles loading the raw dataset, parsing text-formatted columns, removing outliers, and producing a clean dataset ready for EDA and modeling.

**Dataset**: `data/house_prices.csv` (15,327 listings from ikman.lk)

In [1]:
import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')

# Paths
DATA_DIR = os.path.join('..', 'data')
RAW_DATA = os.path.join(DATA_DIR, 'house_prices.csv')
CLEANED_DATA = os.path.join(DATA_DIR, 'cleaned_data.csv')

## 1. Load Raw Data

In [2]:
df = pd.read_csv(RAW_DATA)
print(f'Raw dataset shape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
df.head(3)

Raw dataset shape: (15327, 17)

Columns: ['Title', 'Sub_title', 'Price', 'Address', 'Baths', 'Land size', 'Beds', 'House size', 'Location', 'Description', 'Post_URL', 'Seller_name', 'Seller_type', 'published_date', 'Geo_Address', 'Lat', 'Lon']


,Title,Sub_title,Price,Address,Baths,Land size,Beds,House size,Location,Description,Post_URL,Seller_name,Seller_type,published_date,Geo_Address,Lat,Lon
0,House with Land for Sale in Matara for sale,"Posted on 06 Nov 2:32 pm, Matara City, Matara","Rs 5,400,000","Gangodagama Roard,Hakmana,Matara.",1,50.0 perches,3,"1,600.0 sqft","Matara City, Matara","Land for sale with house Matara,Hakmana Gangod...",https://lankapropertyweb.com/en/ad/house-with-...,Ishara Dilshan,Member,2021-11-06 14:32:00,"Matara City, Matara, Sri Lanka",80.500000,6.166670
1,à¶ à¶½à·à¶­à· à¶à·à¶¸à¶» 3 à¶ à¶à·à· à...,"Posted on 24 Oct 7:27 am, Athurugiriya, Colombo","Rs 16,800,000",Athurugiriya Galwarusapare,3,8.0 perches,3,"1,480.0 sqft","Athurugiriya, Colombo",*House For Sale In Athurugiriya *Galwarusapare...,https://lankapropertyweb.com/en/ad/alut-kaamr-...,DILRUWAN REAL ESTATE,Premium-Member,2021-10-24 07:27:00,"Athurugiriya, Colombo, Sri Lanka",79.989929,6.877246
2,Kelaniya - House on 20P Land for sale for sale,"Posted on 17 Nov 5:19 pm, Kelaniya, Gampaha","Rs 20,000,000",Kelaniya- Ranaviru Maldeniya Road,2,20.0 perches,3,"2,800.0 sqft","Kelaniya, Gampaha","Kelaniya- Biyagama road, Ranaviru Maladeniya R...",https://lankapropertyweb.com/en/ad/kelaniya-ho...,Provident Paradise (Pvt) Ltd,Member,2021-11-17 17:19:00,"Kelaniya, Gampaha, Sri Lanka",79.914926,6.951178


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15327 entries, 0 to 15326
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Title           15327 non-null  object 
 1   Sub_title       15327 non-null  object 
 2   Price           15327 non-null  object 
 3   Address         11191 non-null  object 
 4   Baths           15327 non-null  object 
 5   Land size       15327 non-null  object 
 6   Beds            15327 non-null  object 
 7   House size      15327 non-null  object 
 8   Location        15327 non-null  object 
 9   Description     15327 non-null  object 
 10  Post_URL        15327 non-null  object 
 11  Seller_name     15327 non-null  object 
 12  Seller_type     15327 non-null  object 
 13  published_date  15327 non-null  object 
 14  Geo_Address     15327 non-null  object 
 15  Lat             15327 non-null  float64
 16  Lon             15327 non-null  float64
dtypes: float64(2), object(15)
memor

In [4]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

Missing values per column:
Title                0
Sub_title            0
Price                0
Address           4136
Baths                0
Land size            0
Beds                 0
House size           0
Location             0
Description          0
Post_URL             0
Seller_name          0
Seller_type          0
published_date       0
Geo_Address          0
Lat                  0
Lon                  0
dtype: int64

Total missing values: 4136


## 2. Parse Price (Target Variable)

Prices are in text format like `Rs 5,400,000`. Need to extract the numeric value.

In [5]:
def parse_price(price_str):
    """Convert 'Rs 5,400,000' to float 5400000.0"""
    if pd.isna(price_str):
        return np.nan
    cleaned = str(price_str)
    cleaned = re.sub(r'^Rs\.?\s*', '', cleaned, flags=re.IGNORECASE)
    cleaned = cleaned.replace(',', '').strip()
    match = re.search(r'(\d+(?:\.\d+)?)', cleaned)
    if match:
        return float(match.group(1))
    return np.nan

df['Price_LKR'] = df['Price'].apply(parse_price)
print(f'Parsed prices: {df["Price_LKR"].notna().sum()} valid out of {len(df)}')
print(f'\nSample parsed prices:')
df[['Price', 'Price_LKR']].head(5)

Parsed prices: 15327 valid out of 15327

Sample parsed prices:


,Price,Price_LKR
0,"Rs 5,400,000",5400000.0
1,"Rs 16,800,000",16800000.0
2,"Rs 20,000,000",20000000.0
3,"Rs 187,000,000",187000000.0
4,"Rs 1,300,000",1300000.0


## 3. Parse Bedrooms & Bathrooms

Values are stored as strings (`'3'`, `'10+'`). Need to convert to integers.

In [6]:
def parse_beds_baths(val):
    """Convert '3' or '10+' to int"""
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    if val == '10+':
        return 10
    match = re.search(r'(\d+)', val)
    return int(match.group(1)) if match else np.nan

df['Bedrooms'] = df['Beds'].apply(parse_beds_baths)
df['Bathrooms'] = df['Baths'].apply(parse_beds_baths)

print('Bedrooms distribution:')
print(df['Bedrooms'].value_counts().sort_index())
print(f'\nBathrooms distribution:')
print(df['Bathrooms'].value_counts().sort_index())

Bedrooms distribution:
Bedrooms
1      110
2      928
3     4222
4     6255
5     2529
6      748
7      204
8      168
9       68
10      95
Name: count, dtype: int64

Bathrooms distribution:
Bathrooms
1     2174
2     3687
3     4134
4     4049
5      766
6      191
7      159
8      130
9       28
10       9
Name: count, dtype: int64


## 4. Parse Land Size (Perches)

In [7]:
def parse_land_size(val):
    """Convert '50.0 perches' to float 50.0"""
    if pd.isna(val):
        return np.nan
    val = str(val)
    match = re.search(r'([\d,]+(?:\.\d+)?)\s*(?:perch|perches|p)', val, re.IGNORECASE)
    if match:
        return float(match.group(1).replace(',', ''))
    return np.nan

df['Land_Size_Perches'] = df['Land size'].apply(parse_land_size)
print(f'Parsed land sizes: {df["Land_Size_Perches"].notna().sum()} valid')
df[['Land size', 'Land_Size_Perches']].head(5)

Parsed land sizes: 15309 valid


,Land size,Land_Size_Perches
0,50.0 perches,50.0
1,8.0 perches,8.0
2,20.0 perches,20.0
3,22.0 perches,22.0
4,6.3 perches,6.3


## 5. Parse House Size (Sq.ft)

In [8]:
def parse_house_size(val):
    """Convert '1,600.0 sqft' to float 1600.0"""
    if pd.isna(val):
        return np.nan
    val = str(val)
    match = re.search(r'([\d,]+(?:\.\d+)?)\s*(?:sq\.?\s*ft|sqft)', val, re.IGNORECASE)
    if match:
        return float(match.group(1).replace(',', ''))
    return np.nan

df['House_Size_Sqft'] = df['House size'].apply(parse_house_size)
print(f'Parsed house sizes: {df["House_Size_Sqft"].notna().sum()} valid')
df[['House size', 'House_Size_Sqft']].head(5)

Parsed house sizes: 15327 valid


,House size,House_Size_Sqft
0,"1,600.0 sqft",1600.0
1,"1,480.0 sqft",1480.0
2,"2,800.0 sqft",2800.0
3,"4,000.0 sqft",4000.0
4,900.0 sqft,900.0


## 6. Parse Location (City & District)

In [9]:
def parse_city(loc):
    if pd.isna(loc):
        return 'Unknown'
    parts = str(loc).split(',')
    return parts[0].strip() if parts else 'Unknown'

def parse_district(loc):
    if pd.isna(loc):
        return 'Unknown'
    parts = str(loc).split(',')
    return parts[-1].strip() if len(parts) > 1 else 'Unknown'

df['City'] = df['Location'].apply(parse_city)
df['District'] = df['Location'].apply(parse_district)

print(f'Unique cities: {df["City"].nunique()}')
print(f'Unique districts: {df["District"].nunique()}')
print(f'\nTop 10 districts:')
print(df['District'].value_counts().head(10))

Unique cities: 172
Unique districts: 24

Top 10 districts:
District
Colombo         10977
Gampaha          2908
Galle             336
Kalutara          309
Kandy             247
Kurunegala        136
Matara             77
Kegalle            47
Ratnapura          37
Anuradhapura       37
Name: count, dtype: int64


## 7. Remove Invalid / Missing Data

In [10]:
print(f'Before cleaning: {len(df)} rows')

# Remove rows with missing/zero prices
df = df[df['Price_LKR'] > 0].copy()
print(f'After removing zero/missing prices: {len(df)}')

# Remove rows with missing key features
df = df.dropna(subset=['Bedrooms', 'Bathrooms', 'Land_Size_Perches', 'House_Size_Sqft'])
print(f'After removing missing features: {len(df)}')

Before cleaning: 15327 rows
After removing zero/missing prices: 15327
After removing missing features: 15309


## 8. Remove Outliers

In [11]:
# Price outliers (1st-99th percentile)
q1 = df['Price_LKR'].quantile(0.01)
q99 = df['Price_LKR'].quantile(0.99)
print(f'Price range before: {df["Price_LKR"].min():,.0f} - {df["Price_LKR"].max():,.0f}')
df = df[(df['Price_LKR'] >= q1) & (df['Price_LKR'] <= q99)]
print(f'After price outlier removal (1%-99%): {len(df)} rows')
print(f'Price range after: {df["Price_LKR"].min():,.0f} - {df["Price_LKR"].max():,.0f}')

# Land size outliers
df = df[df['Land_Size_Perches'] <= 500]
print(f'After land size filter (<=500p): {len(df)}')

# House size outliers
df = df[df['House_Size_Sqft'] <= 20000]
print(f'After house size filter (<=20000sqft): {len(df)}')

# Bedrooms & Bathrooms sanity
df = df[(df['Bedrooms'] >= 1) & (df['Bedrooms'] <= 10)]
df = df[(df['Bathrooms'] >= 1) & (df['Bathrooms'] <= 10)]
print(f'After bedroom/bathroom filter (1-10): {len(df)}')

Price range before: 3,600 - 1,900,000,000
After price outlier removal (1%-99%): 15043 rows
Price range after: 3,500,000 - 190,000,000
After land size filter (<=500p): 15024
After house size filter (<=20000sqft): 15003
After bedroom/bathroom filter (1-10): 15003


## 9. Select Final Features & Save

In [12]:
feature_columns = ['Bedrooms', 'Bathrooms', 'Land_Size_Perches', 'House_Size_Sqft', 'City', 'District']
target_column = 'Price_LKR'

df_clean = df[feature_columns + [target_column]].copy()
df_clean = df_clean.reset_index(drop=True)

print(f'Final cleaned dataset shape: {df_clean.shape}')
print(f'Features: {feature_columns}')
print(f'Target: {target_column}')

df_clean.head(10)

Final cleaned dataset shape: (15003, 7)
Features: ['Bedrooms', 'Bathrooms', 'Land_Size_Perches', 'House_Size_Sqft', 'City', 'District']
Target: Price_LKR


,Bedrooms,Bathrooms,Land_Size_Perches,House_Size_Sqft,City,District,Price_LKR
0,3,1,50.00,1600.0,Matara City,Matara,5400000.0
1,3,3,8.00,1480.0,Athurugiriya,Colombo,16800000.0
2,3,2,20.00,2800.0,Kelaniya,Gampaha,20000000.0
3,5,5,22.00,4000.0,Colombo 6,Colombo,187000000.0
4,4,4,11.00,3300.0,Talawatugoda,Colombo,55000000.0
5,4,4,10.00,4100.0,Kandy City,Kandy,45000000.0
6,3,3,9.00,1400.0,Piliyandala,Colombo,16700000.0
7,4,4,10.00,4000.0,Talawatugoda,Colombo,50000000.0
8,4,2,10.00,5000.0,Gampaha City,Gampaha,19500000.0
9,3,1,9.85,1800.0,Katugastota,Kandy,11500000.0


In [13]:
df_clean.describe()

,Bedrooms,Bathrooms,Land_Size_Perches,House_Size_Sqft,Price_LKR
count,15003.000000,15003.000000,15003.000000,15003.000000,1.500300e+04
mean,3.986203,2.976805,13.212529,2703.757594,3.317354e+07
std,1.231560,1.335374,14.282121,1436.572377,2.847001e+07
min,1.000000,1.000000,0.000000,0.000000,3.500000e+06
25%,3.000000,2.000000,7.800000,1650.000000,1.650000e+07
50%,4.000000,3.000000,10.000000,2500.000000,2.600000e+07
75%,4.000000,4.000000,14.000000,3250.000000,3.800000e+07
max,10.000000,10.000000,480.000000,20000.000000,1.900000e+08


In [14]:
# Save cleaned data
df_clean.to_csv(CLEANED_DATA, index=False)
print(f'Saved cleaned data to: {CLEANED_DATA}')
print(f'Shape: {df_clean.shape}')

Saved cleaned data to: ..\data\cleaned_data.csv
Shape: (15003, 7)
